# Module 7 Assignment: Rebuild Your Persona Agent on the OpenAI Agents SDK

Lab 6 rebuilt Study Buddy on top of the OpenAI Agents SDK: same persona, same notes, same three
tools from Lab 5, but with `Agent` and `Runner` replacing the hand written loop and dispatcher.

This assignment makes you do the same move on your **own** Persona Agent. By the end of Module 6
your agent had a persona, study notes, and three tools routed by a `call_tool(name, arguments)`
dispatcher you wrote by hand. Nothing about what your agent *does* needs to change here: you are
porting it to the framework, not redesigning it.

**What you'll build:**
1. An Azure configured `OpenAIChatCompletionsModel`, with tracing disabled, exactly like Lab 6
2. Your persona and notes, carried over from your Module 6 assignment
3. All three of your existing tools, each ported to `@function_tool`
4. One `Agent` that declares your persona, your notes, and all three tools
5. A test using `await Runner.run`
6. The same Gradio chat interface every lab and assignment has used since Lab 4, adapted from Lab 6

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 6 made the same kind of change to each of Study
Buddy's three tools, and do the analogous thing to yours.

## 0. Install the packages (run once)

Same as Lab 6.

In [1]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

Resolved 79 packages in 974ms
 Downloaded pydantic-core
 Downloaded numpy
 Downloaded openai
 Downloaded gradio
Prepared 73 packages in 7.64s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 73 packages in 7.70s
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anyio==4.15.1
 + attrs==26.1.0
 + audioop-lts==0.2.2
 + brotli==1.2.0
 + certifi==2026.7.22
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cryptography==50.0.2
 + fastapi==0.143.1
 + filelock==4.1.2
 + fsspec==2026.9.0
 + gradio==6.30.0
 + gradio-client==2.7.2
 + griffelib==2.3.2
 + groovy==0.1.2
 + h11==0.16.0
 + hf-gradio==0.4.1
 + hf-xet==1.7.0
 + httpcore==1.0.9
 + httpcore2==2.13.1
 + httpx==0.28.1
 + httpx2==2.13.1
 + huggingface-hub==2.2.0
 + idna==3.20
 + jinja2==3.1.6
 + jiter==0.17.0
 + jsonschema==4.26.0
 + jsonschem

## 1. Imports

**TODO:** Copy the imports from Lab 6, Step 1 (including `gradio as gr`), plus `requests` (and
anything else your own tools from the Module 6 assignment need, for example `html` if one of your
tools calls an API that returns HTML-escaped text).

In [3]:
# TODO: copy your imports from Lab 6 (including gradio as gr), plus requests and anything your tools need
from dotenv import load_dotenv
import os
import asyncio
import html
from typing import Literal, Optional

import requests
from openai import AsyncAzureOpenAI
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel, set_tracing_disabled
from pypdf import PdfReader
import gradio as gr


c:\Users\estev\Assignment 7\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load your Azure OpenAI credentials from `.env`

Identical to Lab 6 and every assignment before it.

In [5]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from Lab 6, Step 3, unchanged: call `set_tracing_disabled(True)`, build an
`AsyncAzureOpenAI` client from your endpoint, key, and API version, then wrap it and your deployment
name in `OpenAIChatCompletionsModel`.

In [6]:
# TODO: set_tracing_disabled(True), then build azure_client and azure_model, exactly like Lab 6
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")


Azure model configured for the OpenAI Agents SDK.


## 4. Your persona and notes

**TODO:** Bring over your persona description and your study notes from your Module 6 assignment,
unchanged. If your notes lived in a PDF, reuse the same `load_notes` helper from Lab 6, Step 4.

In [18]:
# TODO: persona = "..." and notes = "..." (or load_notes(...) if you used a PDF), from Module 6
NOTES_PATH = "notes.pdf"

persona = """You're a professional, ethical, and fancy-worded Quiz Game Intructor."""

def load_notes(path):
    try:
        reader = PdfReader(path)
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
        return text.strip()
    except FileNotFoundError:
        return "(notes.pdf not found yet: add it to this folder to enable real study content.)"

study_notes = load_notes(NOTES_PATH)
print("Persona loaded:", persona)
print(study_notes[:400])


Persona loaded: You're a professional, ethical, and fancy-worded Quiz Game Intructor.
types
 
of
 
video
 
games
 
genre:
 
 
-
 
Action
 
-
 
Adventure
 
-
 
RPG
 
-
 
Strategy
 
-
 
Simulation
 
-
 
Sports
 
-
 
Racing
 
-
 
Horror
 
-
 
Puzzle
 
-
 
Fighting
 
-
 
Platformer
 
-
 
Shooter
 
-
 
Survival
 
-
 
Stealth
 
-
 
Sandbox
 
-
 
Rhythm


## 5. Port your first tool

**TODO:** Take your first tool from the Module 6 assignment and rewrite it as a plain, typed Python
function decorated with `@function_tool`, the same way Lab 6, Step 5a ported
`log_missed_question_tool`. Keep the function's actual behavior identical: only the decorator and
the removal of your hand written JSON schema should change.

In [11]:
# TODO: your first tool, decorated with @function_tool

LOG_PATH = "missed_questions.txt"

@function_tool
def my_tool(value: str) -> str:
    """Log a missed question so the student can review it later.

    Args:
        value: The question or text to save for review.
    """
    print(f"Tool called to log a missed question: {value}")

    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(value + "\n")

    return "the missed questions is for review"

print("Tool ready:", my_tool.name)

Tool ready: my_tool


## 6. Port your second tool

**TODO:** Same idea, for your second tool (in Module 6 this was the one backed by a free, no-key
external API), the same way Lab 6, Step 5b ported `fetch_bonus_question_tool`.

In [12]:
# TODO: your second tool, decorated with @function_tool

@function_tool
def api_tool(value: str) -> str:
    """Fetch a bonus trivia question from a free external API.

    Args:
        value: The requested trivia category or topic.
    """
    print(f"Tool called to fetch a bonus trivia question: {value}")

    params = {"amount": 1, "type": "multiple", "category": 15}

    try:
        response = requests.get(
            "https://opentdb.com/api.php",
            params=params,
            timeout=20
        )
        response.raise_for_status()
        data = response.json()

    except (requests.exceptions.RequestException, ValueError) as e:
        print(f"api_tool network error: {e}")
        return "The trivia API is temporarily unavailable. Skip the bonus question and continue the quiz."

    if data.get("response_code") != 0 or not data.get("results"):
        return "Could not fetch a bonus question right now. Skip it and continue the quiz."

    q = data["results"][0]
    question = html.unescape(q["question"])
    answer = html.unescape(q["correct_answer"])

    return f"Bonus question: {question} (correct answer: {answer})"

print("Tool ready:", api_tool.name)

Tool ready: api_tool


## 7. Port your third tool

**TODO:** Same idea, for your third, self-designed tool, the same way Lab 6, Step 5c ported
`check_progress_tool`.

In [13]:
# TODO: your third tool, decorated with @function_tool

@function_tool
def third_tool(value: str) -> str:
    """Record a numerical quiz score in a file for later review.

    Args:
        value: The numerical quiz score to record.
    """
    try:
        score = int(value)

        with open("score.txt", "a", encoding="utf-8") as f:
            f.write(f"{score}\n")

        print(f"Tool called to record score: {score}")

        return f"Score of {score} was successfully recorded."

    except ValueError:
        return "Please provide a valid numerical score."

print("Tool ready:", third_tool.name)

Tool ready: third_tool


## 8. Build your Agent

**TODO:** Build one `Agent`, the same way Lab 6, Step 6 built Study Buddy: a `name`, `instructions`
built from your persona and your notes (with the same rules your old system prompt had about when
to use each tool), a `tools` list containing all three of your ported tools, and `model=azure_model`.

Notice what is missing compared to your Module 6 assignment: no `call_tool(name, arguments)`
dispatcher, and no JSON schema for any of the three tools. `Agent` and `@function_tool` already
cover both.

In [ ]:
# TODO: instructions = f"""..."""  (persona + notes + rules, like Module 6's system prompt)
# TODO: your_agent = Agent(name=..., instructions=instructions, tools=[...], model=azure_model)

instructions = f"""

{persona}
You're rank quiz instructor, you are here to help make this ranking quiz interesting.

use the study notes as the reference for the rank of the video game genres, they are suppose to be displayed randomly and the user suppose
to guess the rank by itself or request a hint to guess it correctly but it can only be used once.

if the student is wrong, type out of the topic or type they don't know, you must record it in my_tool before moving on,
so they can review later.

if the student is asking for a bonus question or another topic you will use api_tool to pull a trivia question from the database.

if the student asks how they are doing you will utilize third_tool to look how many questions they got it right or got it wrong 
and record that in score.txt.

Study notes:
{study_notes}
"""

study_buddy = Agent(
    name="Rank Quiz Instructor",
    instructions=instructions,
    tools=[my_tool, api_tool, third_tool],
    model=azure_model,
)

print("Agent built:", study_buddy.name)


Agent built: Rank Quiz Instructor


## 9. Test it: `await Runner.run`

**TODO:** Run your agent with `await Runner.run(your_agent, "...")`, the same way Lab 6, Step 7
tested Study Buddy. Pick a message that should trigger at least one of your three tools, and
confirm the tool's visible effect (a line appended to a file, a number that changed, or whatever
your tool does) actually happened.

Stick to `await Runner.run(...)` here, not `Runner.run_sync`: Jupyter notebooks already run their
own event loop in the background, and `Runner.run_sync` raises
`RuntimeError: AgentRunner.run_sync() cannot be called when an event loop is already running.` when
called from a notebook cell.

In [15]:
# TODO: result = await Runner.run(your_agent, "...")
# TODO: print(result.final_output)
result = await Runner.run(study_buddy, "Hi! Can you quiz me with one question from my notes?")
print(result.final_output)


Sure — first question:

What is the #1 video game genre in your notes?


## 10. Give it a real chat UI with Gradio

**TODO:** Adapt the `chat` function and `gr.ChatInterface` call from Lab 6, Step 8 to your own
agent: an `async def chat(message, history)` function that rebuilds `history` as a plain list of
`{"role": ..., "content": ...}` dictionaries, appends the new `message` as one more `user` turn,
passes that to `await Runner.run(your_agent, ...)`, and returns `result.final_output`, handed to
`gr.ChatInterface(chat).launch(inbrowser=True)`.

As in Lab 6, `gr.ChatInterface` accepts your async `chat` function directly, and its `history`
already comes in the shape `Runner.run` expects, so no extra conversion is needed.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [16]:
# TODO: async def chat(message, history): ... (see Lab 6, Step 8)
# TODO: gr.ChatInterface(chat).launch(inbrowser=True)
async def chat(message, history):
    history = [{"role": item["role"], "content": item["content"]} for item in history]
    result = await Runner.run(study_buddy, history + [{"role": "user", "content": message}])
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. Which parts of your Module 6 assignment did you copy over completely unchanged? Which parts,
   if any, needed a real change rather than just a different decorator?
- my open AI, persona and notes were the only unchanged copies from module 6, the rest needed a change for adapting to the ports and rewrote my agent to be more up to the tools update, but the main difference will be the addition of @function_tool

2. Compare your old `call_tool(name, arguments)` dispatcher to what `Agent` and `Runner` do for you
   now. What exactly is `Runner.run` doing in place of that dispatcher and your `while`/`for` loop?
-

3. Your Module 6 system prompt included sentences telling the model when to use each tool. Did any
   of that wording need to change for the OpenAI Agents SDK, or did it carry over directly into
   `instructions`?
-

4. If you added a fourth tool to this agent, what would you need to write, and what would you not
   need to write, compared to adding a fourth tool by hand back in Module 6?
-


## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm all three of your ported tools are reachable, `tools=[...]` lists all three, and at least
  one tool call actually happened during your test in Step 9 or Step 10.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module7Assignment_JaneDoe.ipynb`, then submit
  it.